<img src="https://raw.githubusercontent.com/aadhilhaq/ADCP_docking/main/brand/banner_wide.png" width="100%" alt="GUI to ADCP Docking">

A graphical interface to [ADCP](https://ccsb.scripps.edu/adcp/), a product of the Scripps Research Institute. Powered by HOPE Lab. Not affiliated with or endorsed by Scripps.

Press ▶ on step 1, then on step 2. The docking runs on this Colab machine, which suits a trial run. For larger searches, install it on a [SLURM cluster](https://github.com/aadhilhaq/ADCP_docking#install-and-run).

<sub>Step 1 downloads ADFRsuite, which contains ADCP, from the Scripps Research Institute. Parts of ADFRsuite are free for academic use only.</sub>

In [ ]:
#@title **1 · Install** &nbsp; <font color="#5f7070">about 5 minutes, once per session</font>
import subprocess, sys, time
from IPython.display import HTML, display

# One status line, updated in place, so the cell ends with a single line.
_status = display(HTML(""), display_id=True)

def _show(text, colour="#5f7070"):
    _status.update(HTML('<div style="font:14px system-ui;color:%s;margin:2px 0">'
                        '%s</div>' % (colour, text)))

def _install():
    steps = [
        ("Installing the interface (1 of 2)",
         [sys.executable, "-m", "pip", "-q", "install", "rdkit", "pdb2pqr",
          "openmm", "pdbfixer",
          "adcp-dock @ git+https://github.com/aadhilhaq/ADCP_docking"]),
        # ADFRsuite comes from the Scripps Research Institute, as they ask,
        # and is checked against a pinned SHA-256 before it is unpacked.
        ("Downloading ADFRsuite from the Scripps Research Institute (2 of 2)",
         [sys.executable, "-m", "adcp_dock.adfr",
          "--dest", "/content/tools/ADFRsuite"]),
    ]
    for label, cmd in steps:
        _show(label + " …")
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode:
            _show("✗ %s failed. Its output is below." % label.split(" (")[0],
                  "#b3412e")
            print((p.stdout + p.stderr)[-4000:])
            return False
    return True

_t0 = time.time()
if _install():
    _m, _s = divmod(int(time.time() - _t0), 60)
    _show(("✓ Installed in %d min %02d s" % (_m, _s)) if _m
          else ("✓ Installed in %d s" % _s), "#1a8a8a")


In [ ]:
#@title **2 · Open GUI to ADCP Docking**
import importlib, importlib.util, os, re, subprocess, sys, time
from IPython.display import HTML, Javascript, display
from google.colab import output

PORT = 8765
RUNS = "/content/adcp_runs"
ADFR = "/content/tools/ADFRsuite"
TOKEN = None

def _note(text, colour="#b3412e"):
    display(HTML('<div style="font:14px system-ui;color:%s;margin:2px 0">%s</div>'
                 % (colour, text)))

importlib.invalidate_caches()
if importlib.util.find_spec("adcp_dock") is None or not os.path.isdir(ADFR):
    _note("Run step 1 first.")
else:
    os.makedirs(RUNS, exist_ok=True)
    # Running this cell again restarts the server instead of adding a second.
    if globals().get("_SERVER") and _SERVER.poll() is None:
        _SERVER.terminate()
        _SERVER.wait(10)
    # ADCP_EXECUTOR=local runs the work on this machine; Colab has no SLURM.
    env = dict(os.environ, ADCP_EXECUTOR="local", ADFRSUITE_HOME=ADFR,
               PATH=ADFR + "/bin:" + os.environ["PATH"])
    _SERVER = subprocess.Popen(
        [sys.executable, "-u", "-m", "adcp_dock.cli", "serve",
         "--host", "127.0.0.1", "--port", str(PORT), "--runs", RUNS],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    seen, deadline = [], time.time() + 60
    while time.time() < deadline and TOKEN is None:
        line = _SERVER.stdout.readline()
        if not line:
            if _SERVER.poll() is not None:
                break
            continue
        seen.append(line)
        m = re.search(r"http://127\.0\.0\.1:\d+/\?t=([A-Za-z0-9_\-]+)", line)
        if m:
            TOKEN = m.group(1)

    if TOKEN is None:
        _note("The interface did not start. Its output is below.")
        print("".join(seen[-30:]))
    else:
        try:
            output.no_vertical_scroll()
        except Exception:
            pass
        output.serve_kernel_port_as_window(
            PORT, path="/?t=" + TOKEN, anchor_text="Open in a separate tab ↗")
        # embed=1 leaves the banner above to this notebook and lets the form
        # use the full width of the frame.
        output.serve_kernel_port_as_iframe(
            PORT, path="/?t=%s&embed=1" % TOKEN, width="100%", height="800")
        # Size the frame to the visible window, so the interface scrolls
        # inside it like an application window.
        display(Javascript("""
(function fit(tries) {
  var f = document.querySelector('iframe');
  if (!f) { if (tries < 100) setTimeout(function () { fit(tries + 1); }, 150); return; }
  f.style.width = '100%';
  f.style.display = 'block';
  f.style.border = '1px solid #d5e0e0';
  f.style.borderRadius = '10px';
  var size = function () {
    f.style.height = Math.max(620, (window.outerHeight || screen.availHeight) - 230) + 'px';
  };
  size();
  window.addEventListener('resize', size);
})(0);
"""))


In [ ]:
#@title **3 · Download a run** &nbsp; <font color="#5f7070">optional: the interface has its own download buttons</font>
RUN = "" #@param {type:"string"}
#@markdown Leave empty for the latest run. The zip includes HOW_TO_CITE.txt, CITATION.bib and CITATION.ris.
import os, shutil
from IPython.display import HTML, display
from google.colab import files

ROOT = "/content/adcp_runs"
runs = sorted((os.path.join(ROOT, d) for d in
               (os.listdir(ROOT) if os.path.isdir(ROOT) else [])
               if os.path.isfile(os.path.join(ROOT, d, "job.json"))),
              key=os.path.getmtime)
if not runs:
    display(HTML('<div style="font:14px system-ui;color:#b3412e">'
                 'No runs yet. Start one in step 2.</div>'))
else:
    pick = os.path.join(ROOT, RUN) if RUN else runs[-1]
    z = shutil.make_archive("/content/" + os.path.basename(pick), "zip", pick)
    display(HTML('<div style="font:14px system-ui;color:#1a8a8a">✓ %s (%.1f MB)</div>'
                 % (os.path.basename(z), os.path.getsize(z) / 1e6)))
    files.download(z)
